# L-1 Phase M：分層聚類 + Map + 熱路徑

從 `company_labels.jsonl`（法人標籤）與 `phase_l_final.csv`（L層事件）：

1. **分層聚類**：依法人標籤向量將公司分群（KMeans k=7）
2. **Map**：PCA 2D 降維視覺化座標
3. **熱路徑**：公司在 L1–L7 間的時序路徑 Top-N 分析

**執行前提**：`company_labels.jsonl` 及 `phase_l_final.csv` 均已存在。

## 0. 安裝套件

In [1]:
!pip install scikit-learn pandas tqdm numpy

## 1. 匯入套件 & 全域設定

In [2]:
import os, json, re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

# ── 路徑設定 ──────────────────────────────────────────
BASE_DIR       = Path(r"D:\yujui\痛點需求地圖")
CORP_JSONL     = BASE_DIR / "corp_label_output" / "company_labels.jsonl"
PHASE_L_CSV    = BASE_DIR / "step3_output"       / "phase_l_final.csv"
OUTPUT_DIR     = BASE_DIR / "phaseM_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 聚類設定 ──────────────────────────────────────────
N_CLUSTERS   = 7      # 分群數（對應 L1–L7 層數）
RANDOM_STATE = 42
TOP_PATH_N   = 30     # 熱路徑顯示前 N 條
MIN_PATH_LEN = 2      # 最短路徑長度（至少 2 個 L 層）

print("設定完成")
print(f"  CORP_JSONL  : {CORP_JSONL}")
print(f"  PHASE_L_CSV : {PHASE_L_CSV}")
print(f"  OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"  N_CLUSTERS={N_CLUSTERS}  TOP_PATH_N={TOP_PATH_N}")

設定完成
  CORP_JSONL  : D:\yujui\痛點需求地圖\corp_label_output\company_labels.jsonl
  PHASE_L_CSV : D:\yujui\痛點需求地圖\step3_output\phase_l_final.csv
  OUTPUT_DIR  : D:\yujui\痛點需求地圖\phaseM_output
  N_CLUSTERS=7  TOP_PATH_N=30


d:\miniforge3\envs\yujui_even\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 子任務 A：載入法人標籤 → 特徵矩陣

從 `company_labels.jsonl` 解析 8 大類標籤，轉換成數值特徵：
- **布林欄位**（外商/家族企業/集團）→ 0/1
- **列表欄位**（競爭者/關注議題等）→ 元素個數
- **文字欄位**（商業模式/員工人數）→ 分類編碼

In [3]:
# ── A1：解析 company_labels.jsonl ─────────────────────

def _bool(v) -> int:
    if v is True:  return 1
    if v is False: return 0
    return 0

def _cnt(v) -> int:
    """list → 元素數；str → 1；None → 0"""
    if isinstance(v, list): return len(v)
    if isinstance(v, str) and v.strip(): return 1
    return 0

def _biz(v) -> int:
    """商業模式：B2B=2, B2C=1, B2B2C/混合=2, 其他=0"""
    if not isinstance(v, str): return 0
    v = v.upper()
    if "B2B" in v: return 2
    if "B2C" in v: return 1
    return 0

def _headcount(v) -> int:
    """員工人數 → 數值"""
    if not isinstance(v, str): return 0
    m = re.search(r'\d+', v)
    return int(m.group()) if m else 0

records = []
with open(CORP_JSONL, encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        lb = r.get('labels', {})
        cid = r.get('company_id', '')

        basic   = lb.get('基本資料', {})
        contact = lb.get('聯絡人', {})
        trade   = lb.get('進出口資訊', {})
        sales   = lb.get('銷售資訊', {})
        group   = lb.get('集團資訊', {})
        pref    = lb.get('偏好及總結', {})
        ind     = lb.get('產業', {})
        chain   = lb.get('產業鏈', {})

        records.append({
            'company_id':     cid,
            'f_foreign':      _bool(basic.get('外商')),
            'f_family':       _bool(basic.get('家族企業')),
            'f_biz_model':    _biz(basic.get('商業模式')),
            'f_headcount':    _headcount(basic.get('員工人數')),
            'f_n_decision':   _cnt(contact.get('決策者')),
            'f_n_family_mbr': _cnt(contact.get('家族企業人員')),
            'f_n_trade_ctry': _cnt(trade.get('國家')),
            'f_n_competitor': _cnt(sales.get('競爭者')),
            'f_group':        _bool(group.get('集團')),
            'f_n_subsidiary': _cnt(group.get('分公司')),
            'f_n_brand':      _cnt(group.get('品牌名稱')),
            'f_n_issues':     _cnt(pref.get('關注議題')),
            'f_industry_ldr': 1 if ind.get('產業龍頭') else 0,
            'f_n_customer':   _cnt(chain.get('法人的客戶')),
            'f_n_supplier':   _cnt(chain.get('法人的供應商')),
        })

corp_df = pd.DataFrame(records)
print(f'公司數：{len(corp_df):,} 家')
print(f'特徵欄位數：{len([c for c in corp_df.columns if c.startswith("f_")]) }')
corp_df.head(3)

公司數：206,817 家
特徵欄位數：15


,company_id,f_foreign,f_family,f_biz_model,f_headcount,f_n_decision,f_n_family_mbr,f_n_trade_ctry,f_n_competitor,f_group,f_n_subsidiary,f_n_brand,f_n_issues,f_industry_ldr,f_n_customer,f_n_supplier
0,,0,0,2,100,13,2,1,3,0,0,0,5,0,4,0
1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0000000001,0,0,1,100,1,0,0,0,0,0,0,2,0,1,0


In [4]:
# ── A2：載入 phase_l_final.csv (L 層事件) ─────────────
phase_df = pd.read_csv(PHASE_L_CSV, encoding='utf-8-sig',
                       usecols=['event_id','company_id','ym','l_layer','source'])
phase_df['ym'] = phase_df['ym'].astype(str)

n_company = phase_df['company_id'].nunique()
print(f'phase_l_final：{len(phase_df):,} 筆  /  {n_company:,} 家公司')
print(phase_df['l_layer'].value_counts().to_string())

C:\Users\DSC\AppData\Local\Temp\ipykernel_51872\337838389.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  phase_df = pd.read_csv(PHASE_L_CSV, encoding='utf-8-sig',


phase_l_final：1,802,590 筆  /  178,790 家公司
l_layer
L1           461267
L5           433565
L2           366442
L7           186410
L3           183624
L6           114567
L4            56679
uncertain        36


---
## 子任務 B：聚類 + PCA Map

1. StandardScaler 標準化特徵矩陣
2. KMeans（k=N_CLUSTERS）分群
3. PCA(n_components=2) 降維 → 2D 座標

In [5]:
# ── B1：特徵矩陣標準化 ─────────────────────────────────
FEAT_COLS = [c for c in corp_df.columns if c.startswith('f_')]
X = corp_df[FEAT_COLS].fillna(0).values.astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'特徵矩陣：{X_scaled.shape}  (公司數 × 特徵數)')
print('特徵欄：', FEAT_COLS)

特徵矩陣：(206817, 15)  (公司數 × 特徵數)
特徵欄： ['f_foreign', 'f_family', 'f_biz_model', 'f_headcount', 'f_n_decision', 'f_n_family_mbr', 'f_n_trade_ctry', 'f_n_competitor', 'f_group', 'f_n_subsidiary', 'f_n_brand', 'f_n_issues', 'f_industry_ldr', 'f_n_customer', 'f_n_supplier']


In [6]:
# ── B2：KMeans 聚類 ─────────────────────────────────────
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
labels = km.fit_predict(X_scaled)
corp_df['cluster'] = labels

print(f'KMeans 完成  k={N_CLUSTERS}')
print('各群人數：')
print(corp_df['cluster'].value_counts().sort_index().to_string())

KMeans 完成  k=7
各群人數：
cluster
0    141323
1     35565
2      2325
3       252
4      2726
5     18784
6      5842


In [7]:
# ── B3：PCA 降維 → 2D Map 座標 ──────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
corp_df['pca_x'] = coords[:, 0]
corp_df['pca_y'] = coords[:, 1]

print(f'PCA 完成  解釋變異：{pca.explained_variance_ratio_.sum():.1%}')
print(f'PC1={pca.explained_variance_ratio_[0]:.1%}  PC2={pca.explained_variance_ratio_[1]:.1%}')

# 各群重心
centroids = corp_df.groupby('cluster')[['pca_x','pca_y']].mean()
print('\n各群 PCA 重心：')
print(centroids.round(3).to_string())

PCA 完成  解釋變異：42.1%
PC1=31.5%  PC2=10.6%

各群 PCA 重心：
         pca_x  pca_y
cluster              
0       -1.207 -0.036
1        1.842 -0.622
2        5.893 -2.630
3        6.645 -5.001
4        3.766 -1.046
5        3.154  2.843
6        3.448 -2.721


In [ ]:
# ── B3-VIZ：PCA 分群散佈圖 + Cluster 特徵雷達圖 ──────────────────────────
import matplotlib
matplotlib.use("Agg")  # 非互動模式，直接存檔
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

OUT_IMG = OUTPUT_DIR / "cluster_pca_map.png"
OUT_RADAR = OUTPUT_DIR / "cluster_radar.png"

COLORS = ["#FF185F","#F97316","#FBBF24","#22C55E","#38BDF8","#A78BFA","#EC4899"]
CLUSTER_NAMES = {
    0: "C0 微型內銷",
    1: "C1 中型活躍",
    2: "C2 大型外向",
    3: "C3 產業龍頭",
    4: "C4 多品牌型",
    5: "C5 家族企業",
    6: "C6 外商型",
}

# ── 散佈圖（PCA Map）──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8), facecolor="#1E293B")
ax.set_facecolor("#0F172A")

sample = corp_df.sample(min(30000, len(corp_df)), random_state=42)
for cid in sorted(sample["cluster"].unique()):
    s = sample[sample["cluster"] == cid]
    ax.scatter(s["pca_x"], s["pca_y"], c=COLORS[cid], s=2, alpha=0.35,
               label=CLUSTER_NAMES.get(cid, f"C{cid}"))

# 標群心
for cid, row in corp_df.groupby("cluster")[["pca_x","pca_y"]].mean().iterrows():
    ax.annotate(CLUSTER_NAMES.get(cid, f"C{cid}"),
               (row["pca_x"], row["pca_y"]),
               fontsize=10, fontweight="bold",
               color="white", ha="center",
               bbox=dict(boxstyle="round,pad=0.3", fc=COLORS[cid], alpha=0.8))

ax.set_title("Phase M：KMeans(k=7) PCA 分群地圖", color="white", fontsize=14, pad=12)
ax.tick_params(colors="#94A3B8")
for spine in ax.spines.values(): spine.set_edgecolor("#334155")
legend = ax.legend(loc="upper right", fontsize=8, framealpha=0.3,
                   labelcolor="white", facecolor="#1E293B")
plt.tight_layout()
fig.savefig(OUT_IMG, dpi=150, bbox_inches="tight")
plt.close()
print(f"PCA Map 已儲存 → {OUT_IMG}")

# ── 特徵雷達圖（各 cluster 均值）─────────────────────────────────
feats = ["f_foreign","f_biz_model","f_headcount","f_n_trade_ctry",
         "f_n_competitor","f_n_brand","f_n_issues","f_n_customer"]
feat_labels = ["外商","商業模式","員工數","貿易國","競爭者","品牌數","議題數","客戶數"]
N = len(feats)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

profiles = corp_df.groupby("cluster")[feats].mean()
# 正規化至 0-1
norm = (profiles - profiles.min()) / (profiles.max() - profiles.min() + 1e-9)

fig2, axes = plt.subplots(2, 4, figsize=(16, 8),
                          subplot_kw=dict(polar=True), facecolor="#1E293B")
fig2.suptitle("各 Cluster 特徵雷達圖", color="white", fontsize=14, y=1.01)

for idx, (cid, row) in enumerate(norm.iterrows()):
    ax2 = axes[idx // 4][idx % 4]
    vals = row[feats].tolist() + [row[feats[0]]]
    ax2.set_facecolor("#0F172A")
    ax2.plot(angles, vals, color=COLORS[cid], linewidth=2)
    ax2.fill(angles, vals, color=COLORS[cid], alpha=0.25)
    ax2.set_xticks(angles[:-1])
    ax2.set_xticklabels(feat_labels, color="#94A3B8", fontsize=8)
    ax2.set_yticklabels([])
    ax2.set_title(CLUSTER_NAMES.get(cid, f"C{cid}"),
                  color="white", fontsize=9, pad=8,
                  bbox=dict(boxstyle="round", fc=COLORS[cid], alpha=0.7))
    ax2.grid(color="#334155")

# 隱藏第 8 個空格
axes[1][3].set_visible(False)
plt.tight_layout()
fig2.savefig(OUT_RADAR, dpi=150, bbox_inches="tight")
plt.close()
print(f"Radar Chart 已儲存 → {OUT_RADAR}")


---
## 子任務 C：熱路徑分析

依公司 ID 將 `phase_l_final.csv` 的事件按年月排序，
提取每家公司的 L 層時序字串（如 `L1→L3→L5`），
統計出現頻率最高的路徑 Top-N。

In [8]:
# ── C1：產生公司 L 層時序路徑 ─────────────────────────
# 每家公司：按 ym 排序 → 取不重複相鄰層序列（壓縮連續相同層）

def compress_path(layers):
    """去除連續重複：[L1,L1,L3,L3,L5] → [L1,L3,L5]"""
    out = []
    for l in layers:
        if not out or l != out[-1]:
            out.append(l)
    return out

grp = phase_df.sort_values(['company_id','ym']).groupby('company_id')

path_counter = Counter()
company_paths = {}

for cid, gdf in tqdm(grp, desc='熱路徑', total=grp.ngroups):
    layers = gdf['l_layer'].tolist()
    path   = compress_path(layers)
    if len(path) >= MIN_PATH_LEN:
        path_str = '→'.join(path)
        path_counter[path_str] += 1
        company_paths[cid] = path_str

print(f'有效公司路徑：{len(company_paths):,} 家')
print(f'不同路徑種類：{len(path_counter):,} 種')

熱路徑: 100%|██████████| 178790/178790 [00:03<00:00, 47068.28it/s]

有效公司路徑：104,758 家
不同路徑種類：47,779 種


In [9]:
# ── C2：Top-N 熱路徑統計 ───────────────────────────────
top_paths = path_counter.most_common(TOP_PATH_N)
path_df = pd.DataFrame(top_paths, columns=['path','count'])
path_df['pct'] = (path_df['count'] / len(company_paths) * 100).round(2)
path_df['n_steps'] = path_df['path'].str.count('→') + 1

print(f'Top-{TOP_PATH_N} 熱路徑（共 {len(company_paths):,} 家）：')
print(path_df.to_string(index=False))

Top-30 熱路徑（共 104,758 家）：
       path  count  pct  n_steps
      L2→L1   4155 3.97        2
      L1→L2   2856 2.73        2
      L5→L1   2784 2.66        2
      L3→L1   1785 1.70        2
   L1→L2→L1   1609 1.54        3
      L1→L3   1518 1.45        2
      L1→L5   1463 1.40        2
      L5→L2   1257 1.20        2
   L1→L5→L1    918 0.88        3
      L2→L5    911 0.87        2
   L1→L3→L1    772 0.74        3
      L4→L1    673 0.64        2
      L6→L1    662 0.63        2
   L5→L2→L1    609 0.58        3
      L1→L4    607 0.58        2
      L2→L3    594 0.57        2
      L1→L6    584 0.56        2
      L5→L3    548 0.52        2
      L3→L2    536 0.51        2
   L2→L1→L2    501 0.48        3
   L2→L5→L1    480 0.46        3
      L7→L1    441 0.42        2
      L3→L5    415 0.40        2
   L1→L6→L1    367 0.35        3
   L5→L2→L5    316 0.30        3
      L5→L7    315 0.30        2
L2→L1→L2→L1    302 0.29        4
      L5→L6    294 0.28        2
   L1→L4→L1    294

---
## 子任務 E：統計報告 & 輸出

In [10]:
# ── E1：各群特徵側寫報告 ─────────────────────────────
print('=' * 60)
print(f'Phase M 聚類報告  k={N_CLUSTERS}')
print('=' * 60)

for cid in sorted(corp_df['cluster'].unique()):
    sub = corp_df[corp_df['cluster'] == cid]
    n   = len(sub)
    print(f'\n── Cluster {cid}（{n} 家）─────────────────────────')
    for col in FEAT_COLS:
        mean = sub[col].mean()
        if mean > 0.05:
            print(f'   {col:<20s} {mean:.2f}')

print()
# 熱路徑 × 聚類分析
corp_df['path'] = corp_df['company_id'].astype(str).map(company_paths)
has_path = corp_df.dropna(subset=['path'])
print(f'\n法人標籤 × 熱路徑 交集：{len(has_path):,} 家')

Phase M 聚類報告  k=7

── Cluster 0（141323 家）─────────────────────────
   f_biz_model          0.06
   f_headcount          8.64
   f_n_decision         0.21
   f_n_competitor       0.06
   f_n_issues           0.23

── Cluster 1（35565 家）─────────────────────────
   f_biz_model          1.57
   f_headcount          82.76
   f_n_decision         1.63
   f_n_trade_ctry       0.33
   f_n_competitor       0.96
   f_n_subsidiary       0.50
   f_n_brand            0.11
   f_n_issues           2.83
   f_n_customer         0.40
   f_n_supplier         0.21

── Cluster 2（2325 家）─────────────────────────
   f_foreign            0.23
   f_family             0.15
   f_biz_model          1.68
   f_headcount          992.95
   f_n_decision         2.47
   f_n_family_mbr       0.12
   f_n_trade_ctry       0.82
   f_n_competitor       1.19
   f_n_subsidiary       1.10
   f_n_brand            0.44
   f_n_issues           3.73
   f_n_customer         0.58
   f_n_supplier         0.42

── Cluster 3（252 家）───

In [11]:
# ── E2：輸出 CSV ──────────────────────────────────────
# 1) company_clusters.csv
CLUSTER_CSV = OUTPUT_DIR / 'company_clusters.csv'
out_cols = ['company_id','cluster','pca_x','pca_y'] + FEAT_COLS + ['path']
corp_df[out_cols].to_csv(CLUSTER_CSV, index=False, encoding='utf-8-sig')
print(f'✅ company_clusters.csv → {CLUSTER_CSV}  ({len(corp_df):,} 行)')

# 2) cluster_profiles.csv  (各群平均特徵)
PROFILE_CSV = OUTPUT_DIR / 'cluster_profiles.csv'
profiles = corp_df.groupby('cluster')[FEAT_COLS].mean().round(3)
profiles.to_csv(PROFILE_CSV, encoding='utf-8-sig')
print(f'✅ cluster_profiles.csv → {PROFILE_CSV}  ({len(profiles)} 群)')

# 3) hotpaths.csv
HOTPATH_CSV = OUTPUT_DIR / 'hotpaths.csv'
path_df.to_csv(HOTPATH_CSV, index=False, encoding='utf-8-sig')
print(f'✅ hotpaths.csv → {HOTPATH_CSV}  ({len(path_df):,} 條路徑)')

print()
print('Phase M 完成！')
print(f'  輸出目錄：{OUTPUT_DIR}')

✅ company_clusters.csv → D:\yujui\痛點需求地圖\phaseM_output\company_clusters.csv  (206,817 行)
✅ cluster_profiles.csv → D:\yujui\痛點需求地圖\phaseM_output\cluster_profiles.csv  (7 群)
✅ hotpaths.csv → D:\yujui\痛點需求地圖\phaseM_output\hotpaths.csv  (30 條路徑)

Phase M 完成！
  輸出目錄：D:\yujui\痛點需求地圖\phaseM_output
